# Phase 7 — Hiệu Chuẩn & Đánh Giá Đối Đầu Checkpoint P0 (Commit `977b282`)

Notebook này phục vụ quy trình đánh giá đối đầu trực tiếp (**Matched Head-to-Head Evaluation**) giữa:
- **`candidate_p0`**: Checkpoint tốt nhất của ứng viên P0 (`pilot_20261109_182359/p0_legacy_full/best.pth`, hash `9d97613c…`).
- **`incumbent_refit`**: Trọng số mô hình hiện hành (`fatformer_anchored_v2_best.pth`, hash `5cd49252…`), được hiệu chuẩn lại (*refit calibration*) độc lập theo cùng phương pháp để đảm bảo so sánh công bằng.

### Yêu cầu trước khi chạy:
1. **Môi trường Colab**: Chọn GPU CUDA (Runtime -> Change runtime type -> T4, L4, hoặc A100).
2. **Google Drive**: Thư mục `MyDrive/Fatformer/` phải có sẵn **5 tệp** bắt buộc:
   - `phase7/improvement_round1/pilot_20261109_182359/p0_legacy_full/best.pth`
   - `checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth`
   - `datasets/test_communityforensics.tar`
   - `datasets/test_benchmark_gans.tar`
   - `datasets/test_benchmark_diffusion.tar`

### Quy trình chạy:
- Chạy tuần tự **Cell 1** (Mount Drive & GPU) -> **Cell 2** (Git Checkout) -> **Cell 3** (Chạy đánh giá Full Benchmark).
- Sau khi hoàn tất, tệp `evaluation_results_without_weights.zip` sẽ được tải về máy. Hãy giải nén vào `wait/phase7_pilot_evaluation/<EVAL_RUN_ID>/`.

In [ ]:
# ==============================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU RUNTIME
# ==============================================================================
from google.colab import drive
import torch

print("[1/2] Đang kết nối Google Drive (vui lòng cấp quyền trong popup nếu được yêu cầu)...")
drive.mount('/content/drive')

print("\n[2/2] Kiểm tra GPU CUDA:")
assert torch.cuda.is_available(), "LỖI: Chưa chọn GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU (T4, L4 hoặc A100) trước khi chạy."
print(f"✓ Đã phát hiện GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# ==============================================================================
# CELL 2: CLONE MÃ NGUỒN TỪ GIT (NHÁNH hoang) & KHÓA COMMIT 977b282
# ==============================================================================
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/fatformer-xla')
PINNED_COMMIT = '977b282'

if not REPO_DIR.exists():
    print("Đang clone repo từ GitHub (nhánh hoang)...")
    !git clone --branch hoang --single-branch https://github.com/LeeVHoangtk3/fatformer-xla.git /content/fatformer-xla
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}. Đang fetch cập nhật...")
    !git -C /content/fatformer-xla fetch origin hoang

print(f"\nKhóa phiên bản tại commit: {PINNED_COMMIT}")
!git -C /content/fatformer-xla checkout {PINNED_COMMIT}

current_commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print(f"\n✓ Commit hiện tại đang chạy: {current_commit}")
assert current_commit.startswith(PINNED_COMMIT), f"Lệch commit! Yêu cầu {PINNED_COMMIT}, thực tế {current_commit}"

In [ ]:
# ==============================================================================
# CELL 3: CHẠY TOÀN BỘ ĐÁNH GIÁ ĐỐI ĐẦU (FULL MATCHED EVALUATION)
# ==============================================================================
# Tiến trình này sẽ tự động:
# 1. Xác thực hash SHA-256 của 5 tệp trên Drive (2 checkpoint + 3 archive dữ liệu).
# 2. Giải nén dữ liệu sang SSD Colab (/content/phase7_p0_evaluation/<EVAL_RUN_ID>/data).
# 3. Trích xuất logits và fit calibration riêng cho từng model trên cùng 400 ảnh val_calib.
# 4. Đóng gói checkpoint, kiểm chứng 1.200 tensor và smoke test 20 ảnh thực tế.
# 5. Chạy benchmark 1.600 ảnh CFE (7 điều kiện) và 82.003 ảnh in-dist (18 tập) cho cả 2 model.
# 6. Đo độ trễ suy luận (latency overhead), tính toán Gate G7.2 và paired comparison.
# 7. Tự động sao lưu kết quả về Drive và tải tệp evaluation_results_without_weights.zip về máy.
# ==============================================================================
from pathlib import Path
import subprocess

REPO = Path('/content/fatformer-xla')
EVAL_REPO = str(REPO)
EVAL_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()

EVAL_STAGE = 'full'
EVAL_MAX_BATCH = 16

print(f"=== BẮT ĐẦU CHẠY GIAI ĐOẠN ĐÁNH GIÁ: {EVAL_STAGE.upper()} ===")
print(f"Commit: {EVAL_COMMIT}")
print(f"Max Batch Size: {EVAL_MAX_BATCH}")

exec(compile(
    (REPO / 'notebooks/task/phase7_pilot_evaluation/start_colab.py').read_text(),
    'start_colab.py', 'exec'
))

print("\n" + "="*80)
print("✓ ĐÁNH GIÁ ĐỐI ĐẦU ĐÃ HOÀN TẤT THÀNH CÔNG!")
print("Tệp evaluation_results_without_weights.zip đã được lưu trên Drive và bắt đầu tải về.")
print("Sau khi tải về, hãy giải nén vào thư mục: wait/phase7_pilot_evaluation/<EVAL_RUN_ID>/")
print("="*80)